In [ ]:
import requests, os
from PIL import Image
from io import BytesIO

base_url = "https://image.tmdb.org/t/p/w342"
poster_dir = "../data/posters/"
os.makedirs(poster_dir, exist_ok=True)

for i, row in movies.iterrows():
    if pd.notna(row['poster_path']):
        url = base_url + row['poster_path']
        try:
            img = Image.open(BytesIO(requests.get(url).content)).resize((224,224))
            img.save(f"{poster_dir}{row['movieId']}.jpg")
        except:
            pass

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.resnet50 import preprocess_input
import numpy as np

base_model = ResNet50(weights='imagenet', include_top=False, pooling='avg')
# Fungsi ekstraksi fitur
def extract_features(img_path):
    img = image.load_img(img_path, target_size=(224,224))
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)
    features = base_model.predict(x, verbose=0)
    return features.flatten()

# Proses semua poster
import glob
poster_files = glob.glob('../data/posters/*.jpg')
visual_embeddings = {}
for f in poster_files:
    movie_id = int(f.split('/')[-1].replace('.jpg',''))
    visual_embeddings[movie_id] = extract_features(f)

# Ubah ke DataFrame: index = movieId, kolom = vektor 2048 dimensi
visual_df = pd.DataFrame.from_dict(visual_embeddings, orient='index')